# A06 · RTB roles, reserves, and censored feedback

Prerequisites: foundations 31–34; A05.

Evidence: **explicitly constructed mechanism experiments, not real ad logs**. This chapter does not claim industrial-system reproduction or online gains.

Learning order: predict → change one variable → run → explain.

> **English companion.** All instructional Markdown is translated. Experiment code, Chinese language examples, execution counts, and saved outputs remain as in the source edition; historical output text may therefore be Chinese. The initial setup cell anchors the working directory and Python import path for this companion's deeper folder; it does not change the experiment. Saved outputs are historical evidence, not a new run. Run from the top after preparing the required data and models.

In [ ]:
# English companions live one directory deeper; keep the original experiment code unchanged.
from pathlib import Path as _TranslationPath
import os as _translation_os
import sys as _translation_sys
_translation_root = next((p for p in [_TranslationPath.cwd(), *_TranslationPath.cwd().parents]
                          if (p / "reclab").is_dir()), None)
if _translation_root is None:
    raise RuntimeError("Open this notebook from inside the Search Recommendation Ads Lab checkout.")
_translation_os.chdir(_translation_root)
if str(_translation_root) not in _translation_sys.path:
    _translation_sys.path.insert(0, str(_translation_root))


## Problem and input/output contract

This is a minimal state machine for protocol roles, not an OpenRTB implementation. A DSP submits a bid; the exchange allocates subject to publisher reserve and competition. Second-price payment=max(reserve, competitor); first-price payment is own bid. Losing despite meeting a public reserve reveals only that the competing price exceeds own bid in this no-ties example, not its exact value. Below-reserve loss reveals no competitor-price lower bound; timeout conveys no such price information. The competitor is not the publisher reserve.

Before running, predict which result the changed variable will affect and which invariants must hold.

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## Run the one-variable comparison

Inputs are small constructed examples visible in source. Output units and constraints are defined above. Run assertions to verify mathematics and accounting.

In [2]:
rows=[ac.rtb(),ac.rtb(reserve=1.3),ac.rtb(mechanism='first'),ac.rtb(timeout=True),ac.rtb(competitor=1.5)]
display(pd.DataFrame(rows).drop(columns='stages'))
assert rows[0]['payment']==1 and rows[1]['payment']==0
assert rows[1]['observed_market_price'] is None
assert rows[3]['lost_price_lower_bound'] is None
assert rows[4]['lost_price_lower_bound']==1.2
assert rows[4]['observed_market_price'] is None

,win,payment,observed_market_price,lost_price_lower_bound
0,True,1.0,1.0,NaN
1,False,0.0,NaN,NaN
2,True,1.2,1.0,NaN
3,False,0.0,NaN,NaN
4,False,0.0,NaN,1.2


## Interpretation and limits

Raising only reserve can turn a previously winning request into no allocation. First-price rules charge differently for the same bid. Winning-price observations alone are not the full market distribution, and timeout is not a price-based loss.

## Inspect source, then try an exercise

Add invalid currency, rejected creative, and duplicate notices to the state machine. Distinguish no-bid, timeout, loss, and no display.

In [3]:
import inspect
print(inspect.getsource(ac.rtb))

def rtb(bid=1.2, competitor=1., reserve=.8, mechanism='second', timeout=False):
    if mechanism not in ('first','second'): raise ValueError('Invalid mechanism')
    win=not timeout and bid>=max(competitor,reserve)
    return dict(win=win, payment=(bid if mechanism=='first' else max(reserve,competitor)) if win else 0.,
                observed_market_price=competitor if win else None,
                lost_price_lower_bound=bid if not win and not timeout and bid>=reserve else None,
                stages=['SSP request','ADX eligibility','DSP response','auction','win/loss notice','feedback'])



## Sources and next step

A-CCR pp. 27–31; A-RTB pp. 18–19 and 38. This is an original Lab exercise, not a copy of the complete lecture text.

[Chinese GitHub chapter](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [Authors’ RTB slides](https://wnzhang.net/slides/cikm-16-rtb-full.pdf). The Chinese deck is not a complete modern advertising-engineering course. Historical descriptions of platforms are not presented as current market facts.

After the exercise, return to the ads learning path and continue to the next semantic unit.